# 🌐 Project 12 — OpenForge: OSS Framework Contribution

**Core Concept:** Contributing a reusable utility to the LangChain ecosystem

### What This Contributes
A PromptTemplateValidator that checks prompt templates for:
- Missing required variables
- Duplicate variables
- Empty templates
- Template length warnings
- Variable naming conventions

### Why This Matters
Every project in this series suffered from prompt template bugs.
This utility prevents them systematically.

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "key"

The Contribution: PromptTemplateValidator

In [3]:
import re
import os
import sys
from dataclasses import dataclass, field
from typing import Optional
from loguru import logger
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Validation Result ────────────────────────────────────────
@dataclass
class ValidationResult:
    is_valid: bool
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    suggestions: list = field(default_factory=list)
    variables_found: list = field(default_factory=list)
    template_length: int = 0

    def display(self):
        print("\n" + "="*55)
        print("PROMPT TEMPLATE VALIDATION RESULT")
        print("="*55)
        print(f"Valid           : {self.is_valid}")
        print(f"Template Length : {self.template_length} chars")
        print(f"Variables Found : {self.variables_found}")

        if self.errors:
            print("\n❌ ERRORS:")
            for error in self.errors:
                print(f"  - {error}")

        if self.warnings:
            print("\n⚠️  WARNINGS:")
            for warning in self.warnings:
                print(f"  - {warning}")

        if self.suggestions:
            print("\n💡 SUGGESTIONS:")
            for suggestion in self.suggestions:
                print(f"  - {suggestion}")

        if self.is_valid and not self.warnings:
            print("\n✅ Template looks good!")
        print("="*55)


# ── PromptTemplateValidator ──────────────────────────────────
class PromptTemplateValidator:
    """
    A reusable utility for validating LangChain prompt templates.
    Catches common issues before they reach the LLM.

    Usage:
        validator = PromptTemplateValidator()
        result = validator.validate(template_str, required_vars=["name", "task"])
        result.display()

    Contribution to: LangChain Community Tools
    Author: Murali Krishna
    """

    def __init__(self,
                 max_template_length: int = 4000,
                 min_template_length: int = 10,
                 warn_template_length: int = 2000):
        self.max_template_length = max_template_length
        self.min_template_length = min_template_length
        self.warn_template_length = warn_template_length
        self.variable_pattern = re.compile(r'\{([^{}]+)\}')
        self.valid_var_pattern = re.compile(r'^[a-zA-Z_][a-zA-Z0-9_]*$')

    def extract_variables(self, template: str) -> list:
        return self.variable_pattern.findall(template)

    def validate(self, template: str,
                 required_vars: Optional[list] = None,
                 forbidden_vars: Optional[list] = None) -> ValidationResult:

        errors = []
        warnings = []
        suggestions = []

        # Check empty template
        if not template or not template.strip():
            return ValidationResult(
                is_valid=False,
                errors=["Template is empty"],
                template_length=0
            )

        template_length = len(template)

        # Check length
        if template_length < self.min_template_length:
            errors.append(f"Template too short ({template_length} chars). Minimum: {self.min_template_length}")

        if template_length > self.max_template_length:
            errors.append(f"Template too long ({template_length} chars). Maximum: {self.max_template_length}")

        if template_length > self.warn_template_length:
            warnings.append(f"Template is long ({template_length} chars). Consider splitting into smaller templates.")

        # Extract variables
        variables = self.extract_variables(template)
        unique_vars = list(set(variables))

        # Check duplicates
        seen = set()
        duplicates = set()
        for var in variables:
            if var in seen:
                duplicates.add(var)
            seen.add(var)

        if duplicates:
            warnings.append(f"Duplicate variables found: {list(duplicates)}. Each variable should appear once.")

        # Check variable naming conventions
        invalid_names = [v for v in unique_vars if not self.valid_var_pattern.match(v)]
        if invalid_names:
            errors.append(f"Invalid variable names: {invalid_names}. Use snake_case alphanumeric names.")

        # Check required variables
        if required_vars:
            missing = [v for v in required_vars if v not in unique_vars]
            if missing:
                errors.append(f"Missing required variables: {missing}")

            extra = [v for v in unique_vars if v not in required_vars]
            if extra:
                warnings.append(f"Extra variables not in required list: {extra}")

        # Check forbidden variables
        if forbidden_vars:
            found_forbidden = [v for v in unique_vars if v in forbidden_vars]
            if found_forbidden:
                errors.append(f"Forbidden variables found: {found_forbidden}")

        # Check for common issues
        if "{{" in template or "}}" in template:
            warnings.append("Found escaped braces {{ or }}. Verify these are intentional.")

        if len(unique_vars) == 0:
            warnings.append("No variables found. Is this template missing dynamic content?")

        if len(unique_vars) > 10:
            warnings.append(f"Many variables ({len(unique_vars)}). Consider breaking into smaller templates.")

        # Suggestions
        if not template.strip().endswith(('.', '?', ':', '!')):
            suggestions.append("Consider ending your prompt with punctuation for clearer instruction.")

        if len(template.split('\n')) == 1 and template_length > 100:
            suggestions.append("Consider adding line breaks for better readability.")

        is_valid = len(errors) == 0

        return ValidationResult(
            is_valid=is_valid,
            errors=errors,
            warnings=warnings,
            suggestions=suggestions,
            variables_found=unique_vars,
            template_length=template_length
        )

    def validate_chat_template(self, messages: list,
                                required_vars: Optional[list] = None) -> ValidationResult:
        full_template = " ".join([
            msg[1] if isinstance(msg, tuple) else str(msg)
            for msg in messages
        ])
        return self.validate(full_template, required_vars)


validator = PromptTemplateValidator()
print("PromptTemplateValidator ready")
print(__doc__ if hasattr(validator, '__doc__') else "Validator initialized")

PromptTemplateValidator ready
Automatically created module for IPython interactive environment


Test Valid Templates

In [4]:
print("========== VALID TEMPLATE TESTS ==========\n")

valid_templates = [
    {
        "name": "Simple QA Template",
        "template": "You are a helpful assistant.\n\nQuestion: {question}\n\nAnswer:",
        "required_vars": ["question"]
    },
    {
        "name": "RAG Template",
        "template": "Answer based on context only.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:",
        "required_vars": ["context", "question"]
    },
    {
        "name": "Agent Template",
        "template": "You are {agent_name}.\n\nTask: {task}\n\nHistory: {history}\n\nRespond:",
        "required_vars": ["agent_name", "task", "history"]
    }
]

for test in valid_templates:
    print(f"Testing: {test['name']}")
    result = validator.validate(
        test["template"],
        required_vars=test.get("required_vars")
    )
    result.display()

========== VALID TEMPLATE TESTS ==========

Testing: Simple QA Template

PROMPT TEMPLATE VALIDATION RESULT
Valid           : True
Template Length : 59 chars
Variables Found : ['question']

✅ Template looks good!
Testing: RAG Template

PROMPT TEMPLATE VALIDATION RESULT
Valid           : True
Template Length : 80 chars
Variables Found : ['question', 'context']

✅ Template looks good!
Testing: Agent Template

PROMPT TEMPLATE VALIDATION RESULT
Valid           : True
Template Length : 65 chars
Variables Found : ['history', 'task', 'agent_name']

✅ Template looks good!


Test Invalid Templates

In [5]:
print("========== INVALID TEMPLATE TESTS ==========\n")

invalid_templates = [
    {
        "name": "Empty Template",
        "template": "",
        "required_vars": ["question"]
    },
    {
        "name": "Missing Required Variable",
        "template": "Answer this: {query}",
        "required_vars": ["question", "context"]
    },
    {
        "name": "Invalid Variable Name",
        "template": "Hello {user-name}, your task is {task}.",
        "required_vars": ["user-name", "task"]
    },
    {
        "name": "Forbidden Variable",
        "template": "Process {user_input} with {api_key}.",
        "required_vars": ["user_input"],
        "forbidden_vars": ["api_key"]
    }
]

for test in invalid_templates:
    print(f"Testing: {test['name']}")
    result = validator.validate(
        test["template"],
        required_vars=test.get("required_vars"),
        forbidden_vars=test.get("forbidden_vars")
    )
    result.display()

========== INVALID TEMPLATE TESTS ==========

Testing: Empty Template

PROMPT TEMPLATE VALIDATION RESULT
Valid           : False
Template Length : 0 chars
Variables Found : []

❌ ERRORS:
  - Template is empty
Testing: Missing Required Variable

PROMPT TEMPLATE VALIDATION RESULT
Valid           : False
Template Length : 20 chars
Variables Found : ['query']

❌ ERRORS:
  - Missing required variables: ['question', 'context']

⚠️  WARNINGS:
  - Extra variables not in required list: ['query']

💡 SUGGESTIONS:
  - Consider ending your prompt with punctuation for clearer instruction.
Testing: Invalid Variable Name

PROMPT TEMPLATE VALIDATION RESULT
Valid           : False
Template Length : 39 chars
Variables Found : ['user-name', 'task']

❌ ERRORS:
  - Invalid variable names: ['user-name']. Use snake_case alphanumeric names.
Testing: Forbidden Variable

PROMPT TEMPLATE VALIDATION RESULT
Valid           : False
Template Length : 36 chars
Variables Found : ['api_key', 'user_input']

❌ ERRORS:
  -

Real World Usage With LangChain

In [6]:
print("========== REAL WORLD USAGE ==========\n")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    api_key=os.environ["GROQ_API_KEY"]
)

def safe_create_chain(system_template: str, human_template: str,
                      required_vars: list):
    print(f"Validating templates before creating chain...")

    system_result = validator.validate(system_template)
    human_result = validator.validate(human_template, required_vars=required_vars)

    if not system_result.is_valid:
        print(f"System template invalid: {system_result.errors}")
        return None

    if not human_result.is_valid:
        print(f"Human template invalid: {human_result.errors}")
        return None

    print("Templates validated successfully — creating chain")
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_template),
        ("human", human_template)
    ])
    return prompt | llm


chain = safe_create_chain(
    system_template="You are a helpful AI assistant. Answer clearly and concisely.",
    human_template="Question: {question}\n\nAnswer:",
    required_vars=["question"]
)

if chain:
    response = chain.invoke({"question": "What is LangChain?"})
    print(f"\nChain response: {response.content[:200]}")

========== REAL WORLD USAGE ==========

Validating templates before creating chain...
Templates validated successfully — creating chain

Chain response: LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs). It provides a set of tools and libraries to simplify the process of integr


OSS Contribution Guide

In [7]:
print("========== OSS CONTRIBUTION GUIDE ==========\n")
print("How to contribute PromptTemplateValidator to LangChain:\n")

steps = [
    ("1. Fork the repo",
     "Go to github.com/langchain-ai/langchain and click Fork"),
    ("2. Clone your fork",
     "git clone https://github.com/MURALIKRISHNAKECE/langchain.git"),
    ("3. Create feature branch",
     "git checkout -b feature/prompt-template-validator"),
    ("4. Add your code",
     "Copy PromptTemplateValidator to langchain/utils/prompt_validator.py"),
    ("5. Write tests",
     "Add tests to tests/unit_tests/test_prompt_validator.py"),
    ("6. Update docs",
     "Add docstring examples and update CHANGELOG.md"),
    ("7. Submit PR",
     "Push branch and open Pull Request with description of changes"),
    ("8. Respond to review",
     "Address reviewer comments and update code as needed")
]

for step, description in steps:
    print(f"{step}")
    print(f"   → {description}\n")

print("PR Description Template:")
print("-" * 40)
print("""
## Summary
Adds PromptTemplateValidator utility for validating LangChain prompt templates
before they reach the LLM.

## Problem
Prompt template bugs are common and only surface at runtime when the LLM call fails.
This utility catches issues at development time.

## Changes
- Added PromptTemplateValidator class to langchain/utils/
- Validates: empty templates, length, variable names, required/forbidden vars
- Returns structured ValidationResult with errors, warnings, suggestions
- Full test coverage in tests/unit_tests/

## Testing
All existing tests pass. New tests added for validator utility.
""")

========== OSS CONTRIBUTION GUIDE ==========

How to contribute PromptTemplateValidator to LangChain:

1. Fork the repo
   → Go to github.com/langchain-ai/langchain and click Fork

2. Clone your fork
   → git clone https://github.com/MURALIKRISHNAKECE/langchain.git

3. Create feature branch
   → git checkout -b feature/prompt-template-validator

4. Add your code
   → Copy PromptTemplateValidator to langchain/utils/prompt_validator.py

5. Write tests
   → Add tests to tests/unit_tests/test_prompt_validator.py

6. Update docs
   → Add docstring examples and update CHANGELOG.md

7. Submit PR
   → Push branch and open Pull Request with description of changes

8. Respond to review
   → Address reviewer comments and update code as needed

PR Description Template:
----------------------------------------

## Summary
Adds PromptTemplateValidator utility for validating LangChain prompt templates
before they reach the LLM.

## Problem
Prompt template bugs are common and only surface at runtime w

Project Summary

In [8]:
print("========== OPENFORGE SUMMARY ==========\n")
print("Project      : OpenForge — OSS Framework Contribution")
print("Author       : K Murali Krishna")
print("Contribution : PromptTemplateValidator for LangChain")
print("\nWhat Was Built:")
print("  ✓ PromptTemplateValidator — validates prompt templates")
print("  ✓ ValidationResult — structured result with errors/warnings")
print("  ✓ Chat template validation support")
print("  ✓ Safe chain creation with pre-validation")
print("\nValidation Checks:")
print("  ✓ Empty template detection")
print("  ✓ Template length validation")
print("  ✓ Variable extraction and naming convention check")
print("  ✓ Required variable enforcement")
print("  ✓ Forbidden variable detection")
print("  ✓ Duplicate variable warning")
print("\nOSS Concepts Demonstrated:")
print("  ✓ Reusable utility design")
print("  ✓ Clean public API with docstrings")
print("  ✓ Structured output pattern")
print("  ✓ Real-world problem solved from portfolio experience")
print("\n12 Projects. 12 Days. Complete.")
print("GitHub: https://github.com/MURALIKRISHNAKECE/AGENTIC_AI-PRODUCTION_GRADE_PROJECTS")

========== OPENFORGE SUMMARY ==========

Project      : OpenForge — OSS Framework Contribution
Author       : K Murali Krishna
Contribution : PromptTemplateValidator for LangChain

What Was Built:
  ✓ PromptTemplateValidator — validates prompt templates
  ✓ ValidationResult — structured result with errors/warnings
  ✓ Chat template validation support
  ✓ Safe chain creation with pre-validation

Validation Checks:
  ✓ Empty template detection
  ✓ Template length validation
  ✓ Variable extraction and naming convention check
  ✓ Required variable enforcement
  ✓ Forbidden variable detection
  ✓ Duplicate variable warning

OSS Concepts Demonstrated:
  ✓ Reusable utility design
  ✓ Clean public API with docstrings
  ✓ Structured output pattern
  ✓ Real-world problem solved from portfolio experience

12 Projects. 12 Days. Complete.
GitHub: https://github.com/MURALIKRISHNAKECE/AGENTIC_AI-PRODUCTION_GRADE_PROJECTS
